# Exploratory Data Analysis (EDA) — Police Shootings Dataset
**Python version of the R assignment work**

This notebook is organized by EDA stages rather than by assignment number. It covers loading and inspecting the data, missing values, cleaning, data types, descriptive statistics, visualizations, contingency tables, correlation, time-based analysis, and clustering.

**How to use this notebook**
1. Open Google Colab.
2. Upload this `.ipynb` file using **File → Upload notebook**.
3. Run the cells from top to bottom.
4. When prompted, upload `shootings.csv`.

Some techniques below (such as replacing missing values with zero, forward fill, and backward fill) are included because they appeared in the assignment work. They are demonstrated on separate copies of the data; they are not automatically treated as the best cleaning choice for this dataset.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
# Upload shootings.csv from your computer.
uploaded = files.upload()

# Use the first uploaded CSV file.
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload a CSV file, preferably shootings.csv.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print("Loaded file:", file_name)
print("Dataset shape:", df.shape)
df.head()

## 3. Initial inspection

In [ ]:
print("First 5 rows:")
display(df.head())

print("\\nLast 5 rows:")
display(df.tail())

print("\\nRandom sample:")
display(df.sample(min(5, len(df)), random_state=123))

print("\\nNumber of rows and columns:")
print(df.shape)

print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nStatistical summary:")
display(df.describe(include="all").T)

## 4. Check missing values

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing Count": missing_by_column,
    "Missing Percent": missing_percent.round(2)
})
display(missing_summary)

print("Total missing values:", int(df.isna().sum().sum()))
print("Rows containing at least one missing value:", int(df.isna().any(axis=1).sum()))
print("Rows with no missing values:", int(df.notna().all(axis=1).sum()))

# Display a few rows that contain missing values.
display(df[df.isna().any(axis=1)].head(10))

### 4A. Visualize missing values

In [ ]:
plt.figure(figsize=(14, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missing Values Across the Dataset")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
missing_percent[missing_percent > 0].sort_values().plot(kind="barh")
plt.title("Percentage of Missing Values by Column")
plt.xlabel("Missing values (%)")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

## 5. Missing-value handling methods

The following cells demonstrate multiple methods from the assignments. Each method starts from a fresh copy of the original data so that the results do not overwrite one another.

In [ ]:
# Method 1: Remove rows containing at least one missing value.
df_drop_rows = df.dropna()
print("Original shape:", df.shape)
print("Shape after dropping incomplete rows:", df_drop_rows.shape)

# Method 2: Remove columns with more than 70% missing values.
missing_pct_original = df.isna().mean() * 100
columns_over_70 = missing_pct_original[missing_pct_original > 70].index.tolist()
df_drop_columns = df.drop(columns=columns_over_70)

print("Columns removed (>70% missing):", columns_over_70)
print("Shape after removing those columns:", df_drop_columns.shape)

In [ ]:
# Method 3: Mean imputation for numeric columns.
df_mean_imputed = df.copy()
numeric_cols = df_mean_imputed.select_dtypes(include=np.number).columns
df_mean_imputed[numeric_cols] = df_mean_imputed[numeric_cols].fillna(
    df_mean_imputed[numeric_cols].mean()
)
print("Missing values remaining after numeric mean imputation:",
      int(df_mean_imputed[numeric_cols].isna().sum().sum()))

# Method 4: Median imputation for numeric columns.
df_median_imputed = df.copy()
numeric_cols_median = df_median_imputed.select_dtypes(include=np.number).columns
df_median_imputed[numeric_cols_median] = df_median_imputed[numeric_cols_median].fillna(
    df_median_imputed[numeric_cols_median].median()
)
print("Missing values remaining after numeric median imputation:",
      int(df_median_imputed[numeric_cols_median].isna().sum().sum()))

In [ ]:
# Method 5: Mode imputation for categorical/text columns.
df_mode_imputed = df.copy()
categorical_cols = df_mode_imputed.select_dtypes(include=["object", "category", "bool"]).columns

for col in categorical_cols:
    modes = df_mode_imputed[col].mode(dropna=True)
    if not modes.empty:
        df_mode_imputed[col] = df_mode_imputed[col].fillna(modes.iloc[0])

print("Missing values remaining in categorical columns:",
      int(df_mode_imputed[categorical_cols].isna().sum().sum()))

# Method 6: Forward fill and backward fill, shown separately.
df_forward_filled = df.copy().ffill()
df_backward_filled = df.copy().bfill()

print("Missing values after forward fill:", int(df_forward_filled.isna().sum().sum()))
print("Missing values after backward fill:", int(df_backward_filled.isna().sum().sum()))

# Method 7: Replace remaining missing values with zero, on a separate copy.
df_missing_zero = df.copy().fillna(0)
print("Missing values after replacing with zero:", int(df_missing_zero.isna().sum().sum()))

**Note:** Mean/median imputation is mainly appropriate for numeric measurements. Mode imputation is often used for categorical values. Forward/backward fill is more appropriate for ordered or time-series data; this dataset represents individual incidents, so carrying values between rows may not be meaningful. Replacing all missing values with zero can create misleading values, especially for age or categorical columns.

## 6. Duplicate records

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Number of duplicate rows:", duplicate_count)

df_no_duplicates = df.drop_duplicates().copy()
print("Shape before removing duplicates:", df.shape)
print("Shape after removing duplicates:", df_no_duplicates.shape)

## 7. Clean column names and standardize text

In [ ]:
df_clean = df_no_duplicates.copy()

# Remove leading/trailing spaces in column names.
df_clean.columns = df_clean.columns.str.strip()

# Normalize text values in object columns: trim extra spaces.
object_cols = df_clean.select_dtypes(include="object").columns
for col in object_cols:
    df_clean[col] = df_clean[col].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Convert text values to lowercase for consistency.
# This changes capitalization only; it does not merge different categories.
for col in object_cols:
    df_clean[col] = df_clean[col].map(
        lambda value: value.lower() if isinstance(value, str) else value
    )

print("Cleaned column names:")
print(df_clean.columns.tolist())
display(df_clean.head())

## 8. Convert data types

In [ ]:
# Convert columns when they exist in the uploaded dataset.
if "id" in df_clean.columns:
    df_clean["id"] = pd.to_numeric(df_clean["id"], errors="coerce")

if "age" in df_clean.columns:
    df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")

# The examples in the assignments use dates like 02/01/15 (day/month/year).
# Try that format first, then use a general parser for values that did not parse.
if "date" in df_clean.columns:
    parsed_date = pd.to_datetime(
        df_clean["date"], format="%d/%m/%y", errors="coerce"
    )
    remaining = parsed_date.isna() & df_clean["date"].notna()
    if remaining.any():
        parsed_date.loc[remaining] = pd.to_datetime(
            df_clean.loc[remaining, "date"], errors="coerce", dayfirst=True
        )
    df_clean["date"] = parsed_date

# Convert common text columns to category dtype where present.
category_candidates = [
    "manner_of_death", "armed", "gender", "race", "city", "state",
    "signs_of_mental_illness", "threat_level", "flee",
    "body_camera", "arms_category"
]
for col in category_candidates:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("category")

# Rename age to Age if present, following the assignment's rename example.
if "age" in df_clean.columns:
    df_clean = df_clean.rename(columns={"age": "Age"})

# The source dataset has a 'date' column rather than a 'year' column.
# Derive Year from date instead of trying to rename a nonexistent column.
if "date" in df_clean.columns and pd.api.types.is_datetime64_any_dtype(df_clean["date"]):
    df_clean["Year"] = df_clean["date"].dt.year

df_clean.info()
display(df_clean.head())

## 9. Create derived columns

In [ ]:
# Create Age_in_5_Years as in the assignment.
if "Age" in df_clean.columns:
    df_clean["Age_in_5_Years"] = df_clean["Age"] + 5
    display(df_clean[["Age", "Age_in_5_Years"]].head())

# Create date-related columns for time-series analysis.
if "date" in df_clean.columns and pd.api.types.is_datetime64_any_dtype(df_clean["date"]):
    df_clean["Month"] = df_clean["date"].dt.month
    df_clean["Month_Name"] = df_clean["date"].dt.month_name()
    df_clean["YearMonth"] = df_clean["date"].dt.to_period("M").astype(str)
    display(df_clean[["date", "Year", "Month", "Month_Name", "YearMonth"]].head())

## 10. Descriptive statistics

In [ ]:
# Select the Age column if available; otherwise use a numeric column for a general summary.
if "Age" in df_clean.columns:
    values = pd.to_numeric(df_clean["Age"], errors="coerce").dropna()
    variable_name = "Age"
else:
    numeric_available = df_clean.select_dtypes(include=np.number).columns.tolist()
    if not numeric_available:
        raise ValueError("No numeric column is available for descriptive statistics.")
    variable_name = numeric_available[0]
    values = pd.to_numeric(df_clean[variable_name], errors="coerce").dropna()

if len(values) > 0:
    summary_stats = pd.DataFrame({
        "Statistic": [
            "Count", "Mean", "Median", "Minimum", "Maximum", "Range",
            "Variance", "Standard Deviation", "Q1", "Q3", "IQR",
            "Skewness", "Kurtosis"
        ],
        "Value": [
            values.count(), values.mean(), values.median(), values.min(),
            values.max(), values.max() - values.min(), values.var(),
            values.std(), values.quantile(0.25), values.quantile(0.75),
            values.quantile(0.75) - values.quantile(0.25),
            values.skew(), values.kurt()
        ]
    })
    print("Descriptive statistics for:", variable_name)
    display(summary_stats)

In [ ]:
# Frequency distribution of manner_of_death.
if "manner_of_death" in df_clean.columns:
    manner_counts = df_clean["manner_of_death"].value_counts(dropna=False)
    display(manner_counts.rename_axis("manner_of_death").reset_index(name="count"))

    manner_counts.plot(kind="bar", figsize=(8, 4))
    plt.title("Frequency of Manner of Death")
    plt.xlabel("Manner of Death")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'manner_of_death' was not found.")

## 11. Univariate analysis

In [ ]:
# Histograms for numeric columns.
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    series = df_clean[col].dropna()
    if len(series) == 0:
        continue
    plt.figure(figsize=(7, 4))
    sns.histplot(series, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# Bar charts for selected categorical columns.
categorical_plot_cols = [
    "manner_of_death", "gender", "race", "threat_level",
    "flee", "body_camera", "signs_of_mental_illness"
]

for col in categorical_plot_cols:
    if col in df_clean.columns:
        counts = df_clean[col].value_counts(dropna=False).head(15)
        plt.figure(figsize=(8, 4))
        sns.barplot(x=counts.index.astype(str), y=counts.values)
        plt.title(f"Frequency of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 12. Boxplots and outlier inspection

In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    series = df_clean[col].dropna()
    if len(series) == 0:
        continue
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=series)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# IQR-based outlier counts for numeric columns.
outlier_rows = []
for col in df_clean.select_dtypes(include=np.number).columns:
    series = df_clean[col].dropna()
    if len(series) < 4:
        continue
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((series < lower_bound) | (series > upper_bound)).sum())
    outlier_rows.append({
        "Column": col,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_count
    })

display(pd.DataFrame(outlier_rows))

## 13. Bivariate and multivariate analysis

In [ ]:
# Compare Age distributions across categories when those columns exist.
if "Age" in df_clean.columns:
    for cat_col in ["gender", "manner_of_death", "race"]:
        if cat_col in df_clean.columns:
            plot_data = df_clean[[cat_col, "Age"]].dropna()
            if not plot_data.empty:
                top_categories = plot_data[cat_col].value_counts().head(10).index
                plot_data = plot_data[plot_data[cat_col].isin(top_categories)]
                plt.figure(figsize=(10, 5))
                sns.boxplot(data=plot_data, x=cat_col, y="Age")
                plt.title(f"Age by {cat_col}")
                plt.xticks(rotation=45, ha="right")
                plt.tight_layout()
                plt.show()

In [ ]:
# Cross-tabulation / contingency tables.
if "gender" in df_clean.columns and "manner_of_death" in df_clean.columns:
    contingency = pd.crosstab(
        df_clean["gender"], df_clean["manner_of_death"],
        margins=True
    )
    print("Contingency table: gender vs manner_of_death")
    display(contingency)

    print("Row-wise percentages:")
    display((pd.crosstab(
        df_clean["gender"], df_clean["manner_of_death"], normalize="index"
    ) * 100).round(2))

In [ ]:
# Scatter plot for numeric variables, if age and id are available.
# id is an identifier, so this plot is only a basic assignment-style example.
if "Age" in df_clean.columns and "id" in df_clean.columns:
    plot_data = df_clean[["id", "Age"]].dropna()
    if not plot_data.empty:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=plot_data, x="id", y="Age", alpha=0.5)
        plt.title("Age vs ID (assignment-style illustration)")
        plt.tight_layout()
        plt.show()

## 14. Correlation analysis

In [ ]:
numeric_df = df_clean.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    correlation_matrix = numeric_df.corr()
    display(correlation_matrix)

    plt.figure(figsize=(8, 6))
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Correlation Matrix of Numeric Columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are needed for correlation analysis.")

## 15. Time-series analysis

In [ ]:
if "date" in df_clean.columns and pd.api.types.is_datetime64_any_dtype(df_clean["date"]):
    date_df = df_clean.dropna(subset=["date"]).copy()
    date_df = date_df.set_index("date").sort_index()

    print("Date range:")
    print(date_df.index.min(), "to", date_df.index.max())

    # Number of records per month.
    monthly_counts = date_df.resample("MS").size()
    display(monthly_counts.rename("Incident Count").to_frame().head())

    plt.figure(figsize=(12, 5))
    monthly_counts.plot()
    plt.title("Number of Records per Month")
    plt.xlabel("Date")
    plt.ylabel("Record Count")
    plt.tight_layout()
    plt.show()

    # Number of records per year.
    yearly_counts = date_df.resample("YS").size()
    display(yearly_counts.rename("Incident Count").to_frame())

    plt.figure(figsize=(10, 4))
    yearly_counts.plot(kind="bar")
    plt.title("Number of Records per Year")
    plt.xlabel("Year")
    plt.ylabel("Record Count")
    plt.tight_layout()
    plt.show()
else:
    print("A usable 'date' column was not found, so time-series analysis was skipped.")

## 16. K-means clustering (k = 2)

The assignment example selects `id` and `age`, scales the features, and runs K-means with two clusters and seed 123. The code below reproduces that approach when the columns exist. Because `id` is an identifier rather than a meaningful measurement, the optional alternative uses Age alone.

In [ ]:
if "id" in df_clean.columns and "Age" in df_clean.columns:
    cluster_data = df_clean[["id", "Age"]].apply(pd.to_numeric, errors="coerce").dropna()

    if len(cluster_data) >= 2:
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(cluster_data)

        kmeans = KMeans(n_clusters=2, random_state=123, n_init=10)
        cluster_labels = kmeans.fit_predict(scaled_features)

        cluster_result = cluster_data.copy()
        cluster_result["Cluster"] = cluster_labels
        print("Cluster counts:")
        display(cluster_result["Cluster"].value_counts().sort_index())

        print("Cluster centers in scaled feature space:")
        display(pd.DataFrame(kmeans.cluster_centers_, columns=["id_scaled", "Age_scaled"]))

        plt.figure(figsize=(8, 5))
        sns.scatterplot(
            data=cluster_result, x="id", y="Age",
            hue="Cluster", palette="Set1", alpha=0.7
        )
        plt.title("K-means Clustering (ID and Age, k=2)")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough complete rows for K-means.")
else:
    print("Both 'id' and 'Age' are required for this assignment-style clustering example.")

### 16A. Optional alternative: cluster by Age only

In [ ]:
if "Age" in df_clean.columns:
    age_data = df_clean[["Age"]].apply(pd.to_numeric, errors="coerce").dropna()

    if len(age_data) >= 2 and age_data["Age"].nunique() >= 2:
        age_scaled = StandardScaler().fit_transform(age_data)
        age_kmeans = KMeans(n_clusters=2, random_state=123, n_init=10)
        age_data = age_data.copy()
        age_data["Cluster"] = age_kmeans.fit_predict(age_scaled)

        display(age_data.groupby("Cluster")["Age"].agg(["count", "mean", "min", "max"]))

        plt.figure(figsize=(8, 4))
        sns.histplot(data=age_data, x="Age", hue="Cluster", multiple="stack")
        plt.title("Age-only K-means Clusters")
        plt.tight_layout()
        plt.show()

## 17. Hierarchical clustering and dendrogram

Hierarchical clustering can be computationally expensive when applied to every row. This cell uses a sample of up to 300 rows to keep the dendrogram practical in Colab. It uses the assignment-style numeric features when available.

In [ ]:
hierarchy_features = [col for col in ["id", "Age"] if col in df_clean.columns]

if len(hierarchy_features) >= 1:
    hierarchy_data = df_clean[hierarchy_features].apply(pd.to_numeric, errors="coerce").dropna()

    if len(hierarchy_data) >= 2:
        # Sampling keeps the dendrogram readable and limits computation.
        sample_size = min(300, len(hierarchy_data))
        hierarchy_sample = hierarchy_data.sample(n=sample_size, random_state=123)

        hierarchy_scaled = StandardScaler().fit_transform(hierarchy_sample)

        linked = linkage(hierarchy_scaled, method="ward")

        plt.figure(figsize=(14, 6))
        dendrogram(linked, truncate_mode="lastp", p=30, leaf_rotation=45, leaf_font_size=10)
        plt.title("Hierarchical Clustering Dendrogram (sample)")
        plt.xlabel("Cluster / Sample Index")
        plt.ylabel("Distance")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough complete rows for hierarchical clustering.")
else:
    print("No suitable numeric features were found.")

## 18. Agglomerative clustering labels

In [ ]:
if len(hierarchy_features) >= 1:
    agg_data = df_clean[hierarchy_features].apply(pd.to_numeric, errors="coerce").dropna()

    if len(agg_data) >= 2:
        sample_size = min(1000, len(agg_data))
        agg_sample = agg_data.sample(n=sample_size, random_state=123)
        agg_scaled = StandardScaler().fit_transform(agg_sample)

        agglomerative = AgglomerativeClustering(n_clusters=2, linkage="ward")
        agg_labels = agglomerative.fit_predict(agg_scaled)

        agg_result = agg_sample.copy()
        agg_result["Cluster"] = agg_labels

        print("Agglomerative cluster counts:")
        display(agg_result["Cluster"].value_counts().sort_index())

        if "Age" in agg_result.columns:
            display(agg_result.groupby("Cluster")["Age"].agg(["count", "mean", "min", "max"]))
    else:
        print("Not enough complete rows for agglomerative clustering.")

## 19. Final cleaned-data checks

In [ ]:
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)
print("Duplicate rows remaining:", int(df_clean.duplicated().sum()))
print("Missing values remaining:", int(df_clean.isna().sum().sum()))

final_missing = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percent": (df_clean.isna().mean() * 100).round(2)
}).sort_values("Missing Count", ascending=False)
display(final_missing)

display(df_clean.head())

## 20. Export cleaned data

In [ ]:
# Download a cleaned copy if you want to use it outside Colab.
output_name = "shootings_cleaned.csv"
df_clean.to_csv(output_name, index=False)
files.download(output_name)

## Quick notes for your video explanation

- Explain each stage in order: inspect → missing values → clean → convert types → summarize → visualize → analyze relationships → time series → clustering.
- Run cells one at a time so you can explain the output before moving on.
- Do not claim that imputation methods are interchangeable. Explain that the notebook demonstrates the methods requested in the assignments.
- In clustering, mention that IDs are identifiers and that the dendrogram uses a sample to avoid excessive computation.